In [1]:
import json
import pandas as pd
import numpy as np
import multiprocessing
import pickle
import os
import gc
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors, AllChem, DataStructs
from mordred import Calculator, descriptors
from tqdm import tqdm

In [2]:
def to_canonical_smiles(smiles):
    if pd.isna(smiles):
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Chem.MolToSmiles(mol, canonical=True) if mol else None
    except Exception as e:
        print(f"SMILES 변환 실패: {smiles}, 오류: {e}")
        return None

def load_foodb_data(csv_file="phytochemical(정제후).csv"):
    # CSV 파일 읽기 (인코딩 지정)
    foodb_df = pd.read_csv(csv_file, encoding='cp949')
    
    # 필요한 컬럼만 선택하고 이름 변경
    foodb_df = foodb_df[['ChemicalID', 'Name', 'name']].copy()
    foodb_df.columns = ['id', 'name', 'raw_SMILES']
    
    # SMILES가 있는 행만 필터링
    foodb_df = foodb_df.dropna(subset=['raw_SMILES'])
    
    # canonical SMILES 변환
    foodb_df['canonical_SMILES'] = foodb_df['raw_SMILES'].apply(to_canonical_smiles)
    
    # canonical_SMILES가 None인 행 제거
    initial_count = len(foodb_df)
    foodb_df = foodb_df.dropna(subset=['canonical_SMILES'])
    removed_count = initial_count - len(foodb_df)
    
    print(f"로드된 화합물 수: {len(foodb_df)}")
    if removed_count > 0:
        print(f"유효하지 않은 SMILES 제거: {removed_count}개")
    
    return foodb_df

def process_molecules(smiles_list, pkl_file='processed_molecules.pkl'):
    if os.path.exists(pkl_file):
        with open(pkl_file, 'rb') as f:
            data = pickle.load(f)
        print(f"불러온 분자 수: {len(data['mols'])}")
        return data['mols'], data['valid_indices']
    
    mols, valid_indices = [], []
    failed_count = 0
    
    for i, smi in enumerate(tqdm(smiles_list, desc="Processing SMILES")):
        if smi is None or smi == '' or pd.isna(smi):
            continue
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                failed_count += 1
                continue
            mol = Chem.AddHs(mol)
            result = AllChem.EmbedMolecule(mol, randomSeed=42)
            if result == 0:
                AllChem.MMFFOptimizeMolecule(mol, maxIters=1000)
                mols.append(mol)
                valid_indices.append(i)
            else:
                failed_count += 1
        except Exception as e:
            failed_count += 1
            continue
    
    with open(pkl_file, 'wb') as f:
        pickle.dump({'mols': mols, 'valid_indices': valid_indices}, f)
    
    print(f"성공: {len(mols)}개, 실패: {failed_count}개, 전체: {len(smiles_list)}개")
    return mols, valid_indices

def get_descriptor_list(ratio='5x'):
    selected_descriptor = pd.read_csv('../descriptor_selection.csv')
    filename = f'descriptors_filtered_FTO_training_{ratio}_ignore3D_False.csv'
    
    mol = Chem.AddHs(Chem.MolFromSmiles('CCO'))
    all_descriptors = list(Calculator(descriptors).descriptors)
    descriptor_list = []
    not_found = []
    
    for desc_name in selected_descriptor[filename].iloc[0:].dropna().tolist():
        found = False
        for desc in all_descriptors:
            if str(desc).endswith(desc_name) or desc_name in str(desc):
                calc = Calculator([desc])
                result = calc(mol)
                print(f"✓ {desc_name} ({desc}): {result[0]}")
                descriptor_list.append(desc)
                found = True
                break
        if not found:
            not_found.append(desc_name)
    
    if not_found:
        print(f"\n경고: 찾지 못한 descriptor ({len(not_found)}개):")
        for desc in not_found:
            print(f"  - {desc}")
    
    return descriptor_list

def calculate_descriptors(mols, descriptor_list, valid_indices, smiles_list):
    if not mols:
        raise ValueError("계산할 분자가 없습니다.")
    
    if not descriptor_list:
        raise ValueError("계산할 descriptor가 없습니다.")
    
    print(f"계산할 descriptor 수: {len(descriptor_list)}")
    print(f"계산할 분자 수: {len(mols)}")
    
    # Calculator 생성
    calc = Calculator(descriptor_list, ignore_3D=False)
    
    # CPU 코어 설정
    total_cores = multiprocessing.cpu_count()
    nproc = max(1, total_cores - 4)
    print(f"사용 CPU 코어: {nproc}/{total_cores}")
    
    # Descriptor 계산
    try:
        desc_df = calc.pandas(mols, nproc=nproc)
    except Exception as e:
        print(f"Descriptor 계산 중 오류 발생: {e}")
        raise
    
    # 유효한 SMILES 추출
    valid_smiles = [smiles_list[i] for i in valid_indices]
    
    # SMILES 길이 확인
    if len(valid_smiles) != len(desc_df):
        print(f"경고: SMILES 수({len(valid_smiles)})와 descriptor 수({len(desc_df)})가 다릅니다.")
    
    # canonical_SMILES 컬럼 추가
    desc_df.insert(0, 'canonical_SMILES', valid_smiles)
    
    # 결과 확인
    print(f"✓ 완료: {len(desc_df)}개 분자의 {len(descriptor_list)}개 descriptor 계산됨")
    print(f"  - 데이터프레임 크기: {desc_df.shape}")
    
    # NaN 값 체크
    nan_count = desc_df.isnull().sum().sum()
    if nan_count > 0:
        print(f"  ⚠ 경고: {nan_count}개의 NaN 값 발견")
    
    return desc_df

def smiles_to_fingerprint(smiles, fp_size=1024, radius=2):
    # None 또는 NaN 체크
    if smiles is None or pd.isna(smiles) or smiles == '':
        return np.zeros(fp_size)
    
    try:
        # SMILES를 분자 객체로 변환
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(fp_size)
        
        # Morgan Fingerprint 생성
        fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(
            mol, 
            radius=radius, 
            nBits=fp_size
        )
        
        # NumPy 배열로 변환
        arr = np.zeros(fp_size, dtype=np.int8)  # 메모리 절약
        DataStructs.ConvertToNumpyArray(fp, arr)
        
        return arr
        
    except Exception as e:
        # 에러 발생 시 zero 배열 반환
        print(f"Fingerprint 생성 실패 ({smiles}): {e}")
        return np.zeros(fp_size)


In [3]:
# 데이터 로드 (함수 활용)
foodb_df = load_foodb_data("phytochemical(정제후).csv")

# SMILES 리스트 추출
smiles_list = foodb_df['canonical_SMILES'].tolist()
print(f"처리할 SMILES 수: {len(smiles_list)}")

# 분자 처리
print("\n=== 분자 3D 구조 생성 중 ===")
mols, valid_indices = process_molecules(smiles_list)

/var/folders/_b/6t34v4kd15j586v3m5stq5x00000gn/T/ipykernel_60936/3964681312.py:13: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  foodb_df = pd.read_csv(csv_file, encoding='cp949')
[00:44:45] SMILES Parse Error: extra open parentheses for input: 'OC[C@H]1[C@H]([C@@H]([C@@H]([C@@H](O1)O[C@](CC)(C'
[00:44:45] SMILES Parse Error: extra open parentheses for input: '[C@@]12(CC[C@H](C[C@@H]1C(=CC=C2)C=O)C'
[00:44:45] Explicit valence for atom # 1 N, 4, is greater than permitted
[00:44:45] SMILES Parse Error: extra open parentheses for input: 'OC[C@H]1[C@H]([C@@H]([C@@H]([C@@H](O1)O[C@](CC)(C'
[00:44:45] SMILES Parse Error: extra open parentheses for input: '[C@@H]1([C@@H]([C@@H]([C@H](O[C@@H]1CO[C@H]1[C@@H]([C@H]([C@@H]([C@H](O1)CO)O)O)O)O[C@@H](C'
[00:44:45] SMILES Parse Error: extra open parentheses for input: '[C@@H]([C@H]([C@@H](O)C'
[00:44:45] Explicit valence for atom # 6 O, 3, is greater than permitted
[00:44:45] SMILES Parse Erro

로드된 화합물 수: 497434
유효하지 않은 SMILES 제거: 724개
처리할 SMILES 수: 497434

=== 분자 3D 구조 생성 중 ===


Processing SMILES:  10%|█         | 50437/497434 [7:34:50<67:11:00,  1.85it/s]  


KeyboardInterrupt: 

In [ ]:
for ratio in ['5x', '10x']:
    print(f"\n{'='*50}")
    print(f"=== Ratio: {ratio} 처리 시작 ===")
    print(f"{'='*50}")
    
    # descriptor 계산
    print(f"\n[1/5] Descriptor 목록 가져오는 중...")
    descriptor_list = get_descriptor_list(ratio)
    print(f"선택된 descriptor 수: {len(descriptor_list)}")
    
    print(f"\n[2/5] Descriptor 계산 중...")
    desc_df = calculate_descriptors(mols, descriptor_list, valid_indices, smiles_list)
    
    # 데이터 병합
    print(f"\n[3/5] 데이터 병합 중...")
    dataset = foodb_df.merge(desc_df, on='canonical_SMILES', how='inner')
    print(f"병합 후 데이터 수: {len(dataset)}")
    
    # 컬럼 분리 (id, name, raw_SMILES, canonical_SMILES가 처음 4개)
    compound_info = dataset.iloc[:, :4]
    descriptor_data = dataset.iloc[:, 4:]
    
    # Fingerprint 추가
    print(f"\n[4/5] Fingerprint 생성 중...")
    fingerprints = compound_info['canonical_SMILES'].apply(smiles_to_fingerprint)
    fingerprints_array = np.vstack(fingerprints.values)
    fp_df = pd.DataFrame(fingerprints_array, columns=[f'X{i+1}' for i in range(fingerprints_array.shape[1])])
    print(f"Fingerprint 생성 완료: {fp_df.shape}")
    
    # 최종 데이터
    print(f"\n[5/5] 최종 데이터 정제 중...")
    final_data = pd.concat([compound_info, descriptor_data, fp_df], axis=1)
    
    # 컬럼명 수정
    final_data.rename(columns={'MATS1pe': 'MATS1p', 'ATSC6pe': 'ATSC6p'}, inplace=True)
    
    # 필요한 컬럼만 선택
    fp_cols, md_cols = check_col_list(ratio)
    final_data = final_data[['id', 'canonical_SMILES'] + fp_cols + md_cols]
    print(f"컬럼 선택 완료: {len(fp_cols)} fingerprints + {len(md_cols)} descriptors")

    # 에러 제거
    data_cols = fp_cols + md_cols
    initial_count = len(final_data)
    
    for col in data_cols:
        if col in final_data.columns:
            string_mask = final_data[col].astype(str).str.contains('missing|error|failed', case=False, na=False)
            if string_mask.any():
                print(f"  ⚠ 컬럼 {col}에서 {string_mask.sum()}개의 에러 발견 및 제거")
                final_data = final_data[~string_mask]
    
    error_removed = initial_count - len(final_data)
    if error_removed > 0:
        print(f"에러 제거: {error_removed}개 행 삭제")

    # 숫자 변환
    final_data[data_cols] = final_data[data_cols].apply(pd.to_numeric, errors='coerce')
    
    # 중복 제거 및 결측치 제거
    before_dedup = len(final_data)
    final_data = final_data.drop_duplicates(subset=['id', 'canonical_SMILES']).reset_index(drop=True)
    duplicates_removed = before_dedup - len(final_data)
    
    before_dropna = len(final_data)
    final_data = final_data.dropna()
    na_removed = before_dropna - len(final_data)

    print(f"\n정제 완료:")
    print(f"  - 중복 제거: {duplicates_removed}개")
    print(f"  - 결측치 제거: {na_removed}개")
    print(f"  - 최종 데이터 수: {len(final_data)}")
    
    # 저장
    output_file = f'filtered_phytochem_{ratio}.csv'
    final_data.to_csv(output_file, index=False)
    print(f"✓ 저장 완료: {output_file}\n")

print("\n" + "="*50)
print("전체 처리 완료!")
print("="*50)

### 뽑힌 케미컬을 데이터에 매칭하기

In [19]:
# 1. select 파일 로드
select = pd.read_excel('../../result/FTO_Final/FTO chemical_26.xlsx')
print(f"Select 데이터: {len(select)}개")
print(f"Select 컬럼: {list(select.columns)}")
select.head()
# 2. foodb_df 확인
print(f"\nFooDB 데이터: {len(foodb_df)}개")
print(f"FooDB 컬럼: {list(foodb_df.columns)}")
foodb_df.head()
# 3. canonical_SMILES 기준으로 매칭
# select에 있는 canonical_SMILES만 foodb_df에서 추출
select_smiles = select['canonical_SMILES'].dropna().unique().tolist()
print(f"\nSelect에 있는 고유 SMILES 수: {len(select_smiles)}")

matched_foodb = foodb_df[foodb_df['canonical_SMILES'].isin(select_smiles)].copy()
print(f"매칭된 FooDB 데이터: {len(matched_foodb)}개")

# 결과 확인
matched_foodb.head()
matched_foodb.to_csv('../../result/FTO_Final/matched_phytochem_FTO26.csv', index=False)
print("\n저장 완료: matched_phytochem_FTO26.csv")

# 1. Select의 canonical_SMILES 추출
select_smiles_set = set(select['canonical_SMILES'].dropna())
print(f"Select의 고유 SMILES 수: {len(select_smiles_set)}")

# 2. FooDB의 canonical_SMILES 추출
foodb_smiles_set = set(foodb_df['canonical_SMILES'].dropna())
print(f"FooDB의 고유 SMILES 수: {len(foodb_smiles_set)}")

# 3. 매칭된 SMILES
matched_smiles = select_smiles_set & foodb_smiles_set
print(f"\n매칭된 SMILES 수: {len(matched_smiles)}")

# 4. 매칭 안 된 SMILES (Select에는 있지만 FooDB에는 없는 것)
unmatched_smiles = select_smiles_set - foodb_smiles_set
print(f"매칭 안 된 SMILES 수: {len(unmatched_smiles)}")

# 5. 매칭 안 된 데이터 확인
if len(unmatched_smiles) > 0:
    print(f"\n⚠️ {len(unmatched_smiles)}개의 SMILES가 FooDB에 없습니다:")
    unmatched_data = select[select['canonical_SMILES'].isin(unmatched_smiles)].copy()
    print(unmatched_data[['canonical_SMILES', 'name']].head(10))  # 처음 10개만 표시
    
    # 매칭 안 된 데이터 저장
    unmatched_data.to_excel('../../result/FTO_Final/unmatched_select_data.xlsx', index=False)
    print("\n매칭 안 된 데이터 저장: unmatched_select_data.csv")
else:
    print("\n✅ 모든 Select 데이터가 FooDB에서 매칭되었습니다!")


Select 데이터: 26개
Select 컬럼: ['연번', 'id', 'public_id', 'name', 'canonical_SMILES', 'probability_5x', 'probability_10x', 'probability_optnc', '비고', 'MoleculeID', 'CATMoS_NT_pred', 'CATMoS_LD50_pred', 'CATMoS_LD50_predRange', 'Cas number', 'Suppliers', '구매가능여부']

FooDB 데이터: 497434개
FooDB 컬럼: ['id', 'name', 'raw_SMILES', 'canonical_SMILES']

Select에 있는 고유 SMILES 수: 26
매칭된 FooDB 데이터: 2936개

저장 완료: matched_phytochem_FTO26.csv
Select의 고유 SMILES 수: 26
FooDB의 고유 SMILES 수: 73663

매칭된 SMILES 수: 19
매칭 안 된 SMILES 수: 7

⚠️ 7개의 SMILES가 FooDB에 없습니다:
                                     canonical_SMILES  \
5                              O=C(O)CNC(=O)c1ccccc1O   
8          O=C(/C=C/c1ccc(O)c(O)c1)Nc1ccc(O)cc1C(=O)O   
12                         CN(CC(=O)O)C(=N)NP(=O)(O)O   
16                              COc1cc(O)ccc1/C=C/C=O   
18          COc1cc(/C=C/C(=O)Nc2ccc(O)cc2C(=O)O)ccc1O   
20        COc1cc2oc3c(O)ccc(O)c3c(=O)c2c(O)c1CC=C(C)C   
24  O=C1O/C(=C(\C(=O)O)c2ccc(O)cc2)C(O)=C1c1cc(O)c...   

    